# Simulatore di Dama con Gymnasium e Stable-Baselines3

Questo notebook implementa un simulatore di dama 8x8, un ambiente `Gymnasium`, un ciclo di training con `stable-baselines3` e una modalita' **Master** basata su ricerca minimax/alpha-beta. La parte RL serve per addestrare e sperimentare; la modalita' piu' forte usa anche ricerca euristica, perche' un modello RL appena allenato non e' automaticamente imbattibile.

Regole implementate: pedine su caselle scure, cattura obbligatoria, catture multiple, promozione a dama, dame a passo singolo diagonale.

In [1]:
# === SEZIONE: INSTALLAZIONE DELLE LIBRERIE ===
# SCOPO: questa cella serve solo se il tuo ambiente Python non ha gia' le librerie necessarie.
# NOTA: la riga con %pip e' commentata, quindi non parte automaticamente.
# Per usarla devi togliere il simbolo # davanti a %pip.


# Esegui questa cella se le librerie non sono gia' installate.
# In Jupyter/Colab puoi togliere il commento alla riga sotto.
# %pip install gymnasium stable-baselines3 numpy matplotlib opencv-python ipywidgets


In [18]:
# === SEZIONE: IMPORT E STRUMENTI DI BASE ===
# SCOPO: caricare librerie e strumenti usati nel resto del notebook.
# dataclass crea oggetti dati semplici. typing rende chiaro il tipo dei parametri.
# numpy gestisce la scacchiera come matrice numerica.
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Sequence, Tuple
import math
import random

import gymnasium as gym
from gymnasium import spaces
import numpy as np

# === SEZIONE: CODIFICA INTERNA DELLE PEDINE ===
# SCOPO: rappresentare ogni casella con un numero.
# 0 indica casella vuota; numeri positivi indicano il bianco; numeri negativi indicano il nero.
EMPTY = 0
WHITE_MAN = 1
WHITE_KING = 2
BLACK_MAN = -1
BLACK_KING = -2

# --- SOTTOSEZIONE: GIOCATORI E PARAMETRI GLOBALI ---
# WHITE e BLACK identificano i due giocatori. BOARD_SIZE imposta la scacchiera 8x8.
# MAX_GAME_STEPS limita la durata per evitare partite infinite.
WHITE = 1
BLACK = -1
BOARD_SIZE = 8
MAX_GAME_STEPS = 220


# === SEZIONE: CLASSE MOVE, CIOE' UNA MOSSA ===
# SCOPO: salvare una mossa in modo strutturato.
# path contiene tutte le caselle attraversate; captures contiene le pedine mangiate.
# frozen=True significa che una mossa, dopo essere stata creata, non viene modificata.
@dataclass(frozen=True)
class Move:
    path: Tuple[Tuple[int, int], ...]
    captures: Tuple[Tuple[int, int], ...] = ()

    # --- SOTTOSEZIONE: CASELLA DI PARTENZA ---
    # OUTPUT: restituisce la prima casella del percorso della mossa.
    @property
    def start(self) -> Tuple[int, int]:
        return self.path[0]

    # --- SOTTOSEZIONE: CASELLA DI ARRIVO ---
    # OUTPUT: restituisce l'ultima casella del percorso della mossa.
    @property
    def end(self) -> Tuple[int, int]:
        return self.path[-1]

    # --- SOTTOSEZIONE: TESTO LEGGIBILE DELLA MOSSA ---
    # LOGICA: usa "-" per mosse normali e "x" per catture, esempio b6xd4.
    def __str__(self) -> str:
        sep = 'x' if self.captures else '-'
        return sep.join(square_name(r, c) for r, c in self.path)


# === SEZIONE: FUNZIONI DI SUPPORTO SULLE CASELLE ===
# SCOPO: validare coordinate, riconoscere caselle giocabili e convertire coordinate in testo.
def inside(row: int, col: int) -> bool:
    return 0 <= row < BOARD_SIZE and 0 <= col < BOARD_SIZE


# PASSAGGIO: una casella e' giocabile solo se e' dentro la scacchiera e sta sulle diagonali scure.
def playable(row: int, col: int) -> bool:
    return inside(row, col) and (row + col) % 2 == 1


# PASSAGGIO: capisce a quale giocatore appartiene una pedina.
def owner(piece: int) -> int:
    return 0 if piece == EMPTY else (WHITE if piece > 0 else BLACK)


# PASSAGGIO: distingue pedina semplice e dama.
def is_king(piece: int) -> bool:
    return abs(piece) == 2


# PASSAGGIO: converte coordinate della matrice in coordinate da scacchiera, tipo a3 o h8.
def square_name(row: int, col: int) -> str:
    return f"{chr(ord('a') + col)}{BOARD_SIZE - row}"


# PASSAGGIO: fa il contrario di square_name, cioe' trasforma "b6" in riga e colonna.
def parse_square(text: str) -> Tuple[int, int]:
    text = text.strip().lower()
    if len(text) != 2 or text[0] < 'a' or text[0] > 'h' or text[1] < '1' or text[1] > '8':
        raise ValueError("Formato casella non valido. Usa per esempio b6 o c3.")
    col = ord(text[0]) - ord('a')
    row = BOARD_SIZE - int(text[1])
    if not playable(row, col):
        raise ValueError("La casella indicata non e' giocabile nella dama.")
    return row, col


# === SEZIONE: SETUP DELLA SCACCHIERA ===
# SCOPO: creare la posizione iniziale della dama con pedine nere sopra e bianche sotto.
def initial_board() -> np.ndarray:
    board = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=np.int8)
    for r in range(3):
        for c in range(BOARD_SIZE):
            if playable(r, c):
                board[r, c] = BLACK_MAN
    for r in range(5, 8):
        for c in range(BOARD_SIZE):
            if playable(r, c):
                board[r, c] = WHITE_MAN
    return board


# --- SOTTOSEZIONE: PROMOZIONE A DAMA ---
# LOGICA: se una pedina arriva all'ultima riga avversaria, diventa dama.
def promote(piece: int, row: int) -> int:
    if piece == WHITE_MAN and row == 0:
        return WHITE_KING
    if piece == BLACK_MAN and row == BOARD_SIZE - 1:
        return BLACK_KING
    return piece


# === SEZIONE: DIREZIONI DI MOVIMENTO ===
# SCOPO: definire in quali diagonali puo' muoversi ogni tipo di pezzo.
def simple_dirs(piece: int) -> List[Tuple[int, int]]:
    if is_king(piece):
        return [(-1, -1), (-1, 1), (1, -1), (1, 1)]
    return [(-1, -1), (-1, 1)] if piece > 0 else [(1, -1), (1, 1)]


# PASSAGGIO: in questa variante, le catture usano le stesse direzioni del movimento semplice.
def capture_dirs(piece: int) -> List[Tuple[int, int]]:
    # Variante italiana semplificata: le pedine catturano solo in avanti, le dame in entrambe le direzioni.
    return simple_dirs(piece)


# === SEZIONE: GERARCHIA DI CATTURA DELLA DAMA ITALIANA ===
# SCOPO: impedire a una pedina semplice di catturare una dama.
# Una dama puo' catturare pedine e dame; una pedina puo' catturare solo pedine.
def can_capture_piece(attacker: int, target: int) -> bool:
    if target == EMPTY:
        return False
    if owner(attacker) == owner(target):
        return False
    if not is_king(attacker) and is_king(target):
        return False
    return True


# === SEZIONE: CLASSE DAMAGAME, MOTORE DELLA PARTITA ===
# SCOPO: contenere scacchiera, turno corrente, cronologia e regole del gioco.
# Questa e' la classe centrale: quasi tutto il resto usa DamaGame.
class DamaGame:
    # --- SOTTOSEZIONE: CREAZIONE DI UNA PARTITA ---
    # INPUT: una scacchiera opzionale e il giocatore che deve muovere.
    # Se board non viene passato, crea la scacchiera iniziale.
    def __init__(self, board: Optional[np.ndarray] = None, current_player: int = WHITE):
        self.board = initial_board() if board is None else board.astype(np.int8).copy()
        self.current_player = current_player
        self.history: List[Move] = []

    # --- SOTTOSEZIONE: COPIA DELLA PARTITA ---
    # SCOPO: simulare mosse future senza modificare la partita vera.
    def clone(self) -> 'DamaGame':
        cloned = DamaGame(self.board, self.current_player)
        cloned.history = list(self.history)
        return cloned

    # --- SOTTOSEZIONE: RICERCA DELLE PEDINE DI UN GIOCATORE ---
    # OUTPUT: riga, colonna e valore di ogni pedina appartenente al player.
    def pieces(self, player: int) -> Iterable[Tuple[int, int, int]]:
        for r in range(BOARD_SIZE):
            for c in range(BOARD_SIZE):
                piece = int(self.board[r, c])
                if owner(piece) == player:
                    yield r, c, piece

    # === SEZIONE: CALCOLO DELLE MOSSE DEL TURNO ===
    # SCOPO: separare catture e mosse semplici.
    # Questo permette due comportamenti:
    # 1. regola rigida: se puoi catturare, puoi scegliere solo catture;
    # 2. regola con soffio: puoi fare una mossa semplice, ma poi perdi la pedina obbligata.
    def capture_moves(self, player: Optional[int] = None) -> List[Move]:
        player = self.current_player if player is None else player
        captures: List[Move] = []
        for r, c, piece in self.pieces(player):
            captures.extend(self._capture_sequences(r, c, piece, self.board, ((r, c),), ()))
        if not captures:
            return []
        # Cattura multipla obbligatoria: tiene solo le sequenze complete piu' lunghe.
        max_len = max(len(m.captures) for m in captures)
        return [m for m in captures if len(m.captures) == max_len]

    def quiet_moves(self, player: Optional[int] = None) -> List[Move]:
        player = self.current_player if player is None else player
        quiet: List[Move] = []
        for r, c, piece in self.pieces(player):
            for dr, dc in simple_dirs(piece):
                nr, nc = r + dr, c + dc
                if playable(nr, nc) and self.board[nr, nc] == EMPTY:
                    quiet.append(Move(((r, c), (nr, nc))))
        return quiet

    def legal_moves(self, player: Optional[int] = None, allow_soffio: bool = False) -> List[Move]:
        player = self.current_player if player is None else player
        captures = self.capture_moves(player)
        quiet = self.quiet_moves(player)
        if captures:
            return captures + quiet if allow_soffio else captures
        return quiet

    # === SEZIONE: CATENE DI CATTURA MULTIPLA ===
    # SCOPO: trovare catture singole e multiple.
    # LOGICA: usa ricorsione, cioe' dopo una cattura richiama se stessa dalla nuova posizione.
    def _capture_sequences(
        self,
        r: int,
        c: int,
        piece: int,
        board: np.ndarray,
        path: Tuple[Tuple[int, int], ...],
        captures: Tuple[Tuple[int, int], ...],
    ) -> List[Move]:
        # PASSAGGIO: found raccoglie tutte le sequenze complete trovate partendo da questa pedina.
        found: List[Move] = []
        for dr, dc in capture_dirs(piece):
            mr, mc = r + dr, c + dc
            lr, lc = r + 2 * dr, c + 2 * dc
            if not (playable(mr, mc) and playable(lr, lc)):
                continue
            middle = int(board[mr, mc])
            if can_capture_piece(piece, middle) and board[lr, lc] == EMPTY:
                # PASSAGGIO: crea una scacchiera temporanea con la cattura applicata.
                # Non modifica subito la board originale, perche' sta solo simulando.
                next_board = board.copy()
                next_board[r, c] = EMPTY
                next_board[mr, mc] = EMPTY
                next_piece = promote(piece, lr)
                next_board[lr, lc] = next_piece
                # PASSAGGIO: controlla se dopo questa cattura ci sono altre catture obbligatorie.
                tails = self._capture_sequences(
                    lr, lc, next_piece, next_board, path + ((lr, lc),), captures + ((mr, mc),)
                )
                if tails:
                    found.extend(tails)
                else:
                    found.append(Move(path + ((lr, lc),), captures + ((mr, mc),)))
        return found

    # === SEZIONE: SOFFIO E APPLICAZIONE DI UNA MOSSA REALE ===
    # SCOPO: applicare la mossa e, se consentito, rimuovere una pedina quando il giocatore ignora una cattura.
    def _apply_soffio(self, move: Move, forced_captures: List[Move]) -> Optional[Tuple[int, int]]:
        if not forced_captures or move.captures:
            return None
        forced_starts = [m.start for m in forced_captures]
        # Se la pedina mossa era proprio una di quelle obbligate a catturare,
        # viene soffiata nella casella di arrivo. Altrimenti viene soffiata
        # la prima pedina che aveva una cattura obbligatoria.
        blown = move.end if move.start in forced_starts else forced_starts[0]
        br, bc = blown
        if self.board[br, bc] != EMPTY:
            self.board[br, bc] = EMPTY
            return blown
        return None

    def apply_move(self, move: Move, allow_soffio: bool = False) -> Optional[Tuple[int, int]]:
        forced_captures = self.capture_moves(self.current_player)
        legal = self.legal_moves(self.current_player, allow_soffio=allow_soffio)
        if move not in legal:
            raise ValueError(f"Mossa illegale: {move}")
        sr, sc = move.start
        er, ec = move.end
        piece = int(self.board[sr, sc])
        self.board[sr, sc] = EMPTY
        for cr, cc in move.captures:
            self.board[cr, cc] = EMPTY
        self.board[er, ec] = promote(piece, er)
        blown = self._apply_soffio(move, forced_captures) if allow_soffio else None
        self.history.append(move)
        self.current_player *= -1
        return blown

    # === SEZIONE: CONTROLLO VINCITORE ===
    # SCOPO: capire se la partita e' finita.
    # Vince chi lascia l'avversario senza pedine o senza mosse disponibili.
    def winner(self) -> Optional[int]:
        white_has = any(owner(int(x)) == WHITE for x in self.board.flat)
        black_has = any(owner(int(x)) == BLACK for x in self.board.flat)
        if not white_has:
            return BLACK
        if not black_has:
            return WHITE
        if not self.legal_moves(self.current_player):
            return -self.current_player
        return None

    # === SEZIONE: RENDER TESTUALE DELLA SCACCHIERA ===
    # SCOPO: stampare la scacchiera come testo, utile per debug e gioco da input.
    def render_text(self) -> str:
        symbols = {EMPTY: '.', WHITE_MAN: 'w', WHITE_KING: 'W', BLACK_MAN: 'b', BLACK_KING: 'B'}
        lines = ['   a b c d e f g h']
        for r in range(BOARD_SIZE):
            rank = BOARD_SIZE - r
            row = ' '.join(symbols[int(self.board[r, c])] if playable(r, c) else ' ' for c in range(BOARD_SIZE))
            lines.append(f'{rank}  {row}  {rank}')
        lines.append('   a b c d e f g h')
        return '\n'.join(lines)


# === SEZIONE: VALUTAZIONE EURISTICA DELLA POSIZIONE ===
# SCOPO: dare un punteggio alla scacchiera per aiutare la macchina a scegliere.
def material_score(board: np.ndarray, player: int) -> float:
    values = {WHITE_MAN: 1.0, WHITE_KING: 2.2, BLACK_MAN: -1.0, BLACK_KING: -2.2}
    raw = sum(values.get(int(x), 0.0) for x in board.flat)
    return raw * player


# --- SOTTOSEZIONE: VALUTAZIONE POSIZIONALE ---
# LOGICA: oltre al materiale, premia centro e avanzamento delle pedine.
def positional_score(board: np.ndarray, player: int) -> float:
    score = material_score(board, player) * 10.0
    for r in range(BOARD_SIZE):
        for c in range(BOARD_SIZE):
            piece = int(board[r, c])
            if owner(piece) == player:
                score += 0.15 * (3.5 - abs(c - 3.5))
                if not is_king(piece):
                    score += 0.08 * ((7 - r) if player == WHITE else r)
            elif owner(piece) == -player:
                score -= 0.15 * (3.5 - abs(c - 3.5))
    return score


# === SEZIONE: IA SEMPLICE BASATA SU EURISTICA ===
# SCOPO: scegliere la mossa che sembra migliore guardando solo la posizione immediata dopo la mossa.
def heuristic_move(game: DamaGame, player: int, rng: Optional[random.Random] = None) -> Optional[Move]:
    rng = rng or random
    moves = game.legal_moves(player)
    if not moves:
        return None
    scored = []
    for move in moves:
        child = game.clone()
        child.current_player = player
        child.apply_move(move)
        score = positional_score(child.board, player) + 3.0 * len(move.captures)
        scored.append((score, rng.random(), move))
    return max(scored, key=lambda x: (x[0], x[1]))[2]


# === SEZIONE: IA MASTER CON MINIMAX E ALPHA-BETA ===
# SCOPO: scegliere una mossa simulando piu' turni futuri.
# depth controlla quanto in profondita' pensa la macchina.
def minimax_move(game: DamaGame, player: int, depth: int = 5) -> Optional[Move]:
    moves = game.legal_moves(player)
    if not moves:
        return None

    # --- SOTTOSEZIONE: FUNZIONE RICORSIVA DI RICERCA ---
    # LOGICA: alterna turno della macchina e turno dell'avversario.
    # alpha e beta tagliano rami inutili della simulazione per velocizzare.
    def search(node: DamaGame, d: int, alpha: float, beta: float) -> float:
        winner = node.winner()
        if winner == player:
            return 10_000 + d
        if winner == -player:
            return -10_000 - d
        if d == 0:
            return positional_score(node.board, player)
        node_moves = node.legal_moves(node.current_player)
        if node.current_player == player:
            value = -math.inf
            ordered = sorted(node_moves, key=lambda m: len(m.captures), reverse=True)
            for move in ordered:
                child = node.clone()
                child.apply_move(move)
                value = max(value, search(child, d - 1, alpha, beta))
                alpha = max(alpha, value)
                if alpha >= beta:
                    break
            return value
        value = math.inf
        ordered = sorted(node_moves, key=lambda m: len(m.captures), reverse=True)
        for move in ordered:
            child = node.clone()
            child.apply_move(move)
            value = min(value, search(child, d - 1, alpha, beta))
            beta = min(beta, value)
            if alpha >= beta:
                break
        return value

    # --- SOTTOSEZIONE: SCELTA FINALE DELLA MOSSA ---
    # PASSAGGIO: valuta ogni mossa possibile e tiene quelle con punteggio massimo.
    best_score = -math.inf
    best_moves: List[Move] = []
    for move in sorted(moves, key=lambda m: len(m.captures), reverse=True):
        child = game.clone()
        child.current_player = player
        child.apply_move(move)
        score = search(child, depth - 1, -math.inf, math.inf)
        if score > best_score:
            best_score = score
            best_moves = [move]
        elif score == best_score:
            best_moves.append(move)
    return random.choice(best_moves)


In [15]:
# === SEZIONE: AMBIENTE GYMNASIUM PER IL REINFORCEMENT LEARNING ===
# SCOPO: trasformare DamaGame in un ambiente compatibile con Stable-Baselines3.
# Il modello RL non usa direttamente la grafica: riceve osservazioni numeriche e produce azioni numeriche.
class DamaEnv(gym.Env):
    """Ambiente Gymnasium: l'agente gioca con il bianco, l'avversario con il nero."""

    # --- SOTTOSEZIONE: METADATI DI VISUALIZZAZIONE ---
    # Dice a Gymnasium che l'ambiente puo' essere renderizzato come testo.
    metadata = {"render_modes": ["human", "ansi"]}

    # --- SOTTOSEZIONE: INIZIALIZZAZIONE AMBIENTE ---
    # Qui vengono definiti spazio azioni, spazio osservazioni, avversario e partita interna.
    def __init__(self, opponent: str = "heuristic", max_steps: int = MAX_GAME_STEPS, seed: Optional[int] = None):
        super().__init__()
        self.action_space = spaces.Discrete(BOARD_SIZE * BOARD_SIZE * BOARD_SIZE * BOARD_SIZE)
        self.observation_space = spaces.Box(low=-2, high=2, shape=(BOARD_SIZE, BOARD_SIZE), dtype=np.int8)
        self.opponent = opponent
        self.max_steps = max_steps
        self.rng = random.Random(seed)
        self.game = DamaGame()
        self.steps = 0

    # === SEZIONE: CODIFICA E DECODIFICA DELLE AZIONI ===
    # SCOPO: trasformare una mossa start/end in un numero singolo, perche' PPO lavora con azioni numeriche.
    @staticmethod
    def encode_action(start: Tuple[int, int], end: Tuple[int, int]) -> int:
        sr, sc = start
        er, ec = end
        return ((sr * BOARD_SIZE + sc) * BOARD_SIZE + er) * BOARD_SIZE + ec

    @staticmethod
    def decode_action(action: int) -> Tuple[Tuple[int, int], Tuple[int, int]]:
        ec = action % BOARD_SIZE
        action //= BOARD_SIZE
        er = action % BOARD_SIZE
        action //= BOARD_SIZE
        sc = action % BOARD_SIZE
        sr = action // BOARD_SIZE
        return (sr, sc), (er, ec)

    # === SEZIONE: MASCHERA DELLE AZIONI LEGALI ===
    # SCOPO: indicare quali azioni numeriche corrispondono a mosse legali nella posizione attuale.
    def legal_action_mask(self) -> np.ndarray:
        mask = np.zeros(self.action_space.n, dtype=bool)
        for move in self.game.legal_moves(WHITE):
            mask[self.encode_action(move.start, move.end)] = True
        return mask

    # === SEZIONE: OSSERVAZIONE DELLO STATO ===
    # SCOPO: restituire la scacchiera in formato numerico, leggibile dal modello.
    def _obs(self) -> np.ndarray:
        return self.game.board.copy()

    # === SEZIONE: RESET DI UN EPISODIO ===
    # SCOPO: iniziare una nuova partita durante training o valutazione.
    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        super().reset(seed=seed)
        if seed is not None:
            self.rng.seed(seed)
        self.game = DamaGame(current_player=WHITE)
        self.steps = 0
        return self._obs(), {"legal_actions": np.flatnonzero(self.legal_action_mask())}

    # === SEZIONE: TRADUZIONE AZIONE MODELLO IN MOSSA ===
    # SCOPO: convertire il numero scelto dal modello in una mossa vera, se e' legale.
    def _select_agent_move(self, action: int) -> Optional[Move]:
        start, end = self.decode_action(int(action))
        candidates = [m for m in self.game.legal_moves(WHITE) if m.start == start and m.end == end]
        if not candidates:
            return None
        return max(candidates, key=lambda m: len(m.captures))

    # === SEZIONE: RISPOSTA AUTOMATICA DELL'AVVERSARIO ===
    # SCOPO: dopo la mossa del bianco/agente, far muovere il nero.
    # L'avversario puo' essere random, heuristic o master.
    def _opponent_move(self) -> None:
        if self.game.winner() is not None:
            return
        if self.opponent == "random":
            moves = self.game.legal_moves(BLACK)
            move = self.rng.choice(moves) if moves else None
        elif self.opponent == "master":
            move = minimax_move(self.game, BLACK, depth=4)
        else:
            move = heuristic_move(self.game, BLACK, self.rng)
        if move is not None:
            self.game.apply_move(move)

    # === SEZIONE: STEP PRINCIPALE DELL'AMBIENTE ===
    # SCOPO: eseguire una mossa dell'agente, una risposta dell'avversario e calcolare reward/fine partita.
    def step(self, action: int):
        self.steps += 1
        # PASSAGGIO: salva il valore materiale prima della mossa per calcolare il reward.
        before = material_score(self.game.board, WHITE)
        move = self._select_agent_move(action)
        if move is None:
            reward = -0.75
            terminated = False
            truncated = self.steps >= self.max_steps
            return self._obs(), reward, terminated, truncated, {"illegal": True, "legal_actions": np.flatnonzero(self.legal_action_mask())}

        self.game.apply_move(move)
        winner = self.game.winner()
        if winner == WHITE:
            return self._obs(), 100.0, True, False, {"move": str(move)}

        # PASSAGGIO: se il bianco non ha gia' vinto, il nero risponde automaticamente.
        self._opponent_move()
        winner = self.game.winner()
        after = material_score(self.game.board, WHITE)
        # PASSAGGIO: reward positivo se migliora il materiale o cattura pezzi.
        # Penalizza le mosse illegali e premia/penalizza molto vittoria o sconfitta.
        reward = (after - before) * 2.5 + len(move.captures) * 2.0
        terminated = winner is not None
        if winner == WHITE:
            reward += 100.0
        elif winner == BLACK:
            reward -= 100.0
        truncated = self.steps >= self.max_steps
        if truncated and not terminated:
            reward += material_score(self.game.board, WHITE)
        info = {"move": str(move), "winner": winner, "legal_actions": np.flatnonzero(self.legal_action_mask())}
        return self._obs(), float(reward), terminated, truncated, info

    # === SEZIONE: RENDER TESTUALE DELL'AMBIENTE ===
    # SCOPO: mostrare la scacchiera durante debug o test.
    def render(self):
        text = self.game.render_text()
        if self.render_mode == "human":
            print(text)
        return text


# === SEZIONE: TEST RAPIDO DELL'AMBIENTE ===
# SCOPO: crea un ambiente, fa reset e stampa la posizione iniziale.
env = DamaEnv(opponent="heuristic")
obs, info = env.reset(seed=42)
print(env.game.render_text())
print("Azioni legali iniziali:", len(info["legal_actions"]))


   a b c d e f g h
8    b   b   b   b  8
7  b   b   b   b    7
6    b   b   b   b  6
5  .   .   .   .    5
4    .   .   .   .  4
3  w   w   w   w    3
2    w   w   w   w  2
1  w   w   w   w    1
   a b c d e f g h
Azioni legali iniziali: 7


In [4]:
# === SEZIONE: CONTROLLO DI COMPATIBILITA' GYMNASIUM ===
# SCOPO: verificare che DamaEnv rispetti le regole richieste da Gymnasium e Stable-Baselines3.
# Se questa cella genera errori, il training PPO potrebbe non funzionare.

# Verifica compatibilita' Gymnasium + Stable-Baselines3.
from stable_baselines3.common.env_checker import check_env

check_env(DamaEnv(opponent="random"), warn=True)
print("Ambiente verificato correttamente.")


Ambiente verificato correttamente.


c:\Users\utente\AppData\Local\Programs\Python\Python313\Lib\site-packages\stable_baselines3\common\env_checker.py:324: UserWarning: Your observation  has an unconventional shape (neither an image, nor a 1D vector). We recommend you to flatten the observation to have only a 1D vector or use a custom policy to properly process the data.
  warnings.warn(


In [5]:
# Training RL. PPO e' stabile per iniziare, ma aumenta total_timesteps per risultati seri.
# === SEZIONE: IMPORT DEL MODELLO DI TRAINING ===
# SCOPO: caricare PPO e Monitor. PPO e' l'algoritmo RL, Monitor registra dati sull'allenamento.
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor

# === SEZIONE: CREAZIONE AMBIENTE DI ALLENAMENTO ===
# SCOPO: far allenare il modello contro un avversario euristico.
train_env = Monitor(DamaEnv(opponent="heuristic"))
# === SEZIONE: CONFIGURAZIONE DEL MODELLO PPO ===
# SCOPO: definire rete, learning rate, numero step e altri iperparametri.
model = PPO(
    "MlpPolicy",
    train_env,
    verbose=1,
    learning_rate=3e-4,
    n_steps=1024,
    batch_size=256,
    gamma=0.98,
)

# Per test rapido: 5_000. Per un training reale: 200_000, 1_000_000 o piu'.
# === SEZIONE: AVVIO TRAINING ===
# SCOPO: far giocare il modello per un certo numero di step.
# NOTA: 5_000 step sono pochi, servono per una demo veloce.
model.learn(total_timesteps=5_000)
# === SEZIONE: SALVATAGGIO MODELLO ===
# SCOPO: salvare il risultato del training in dama_ppo_model.zip.
model.save("dama_ppo_model")
print("Modello salvato in dama_ppo_model.zip")


Using cpu device
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 220      |
|    ep_rew_mean     | -165     |
| time/              |          |
|    fps             | 888      |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 1024     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 220          |
|    ep_rew_mean          | -165         |
| time/                   |              |
|    fps                  | 704          |
|    iterations           | 2            |
|    time_elapsed         | 2            |
|    total_timesteps      | 2048         |
| train/                  |              |
|    approx_kl            | 0.0026159876 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -8.32        |
|    e

In [5]:
# === SEZIONE: FUNZIONE DI VALUTAZIONE DEL MODELLO ===
# SCOPO: far giocare al modello piu' partite e contare i risultati.
def evaluate_model(model, games: int = 20, opponent: str = "heuristic") -> Dict[str, int]:
    # --- SOTTOSEZIONE: CONTATORI RISULTATI ---
    # white_win: vince agente/bianco. black_win: vince avversario/nero. draw_or_truncated: partita non conclusa.
    results = {"white_win": 0, "black_win": 0, "draw_or_truncated": 0}
    # --- SOTTOSEZIONE: CICLO DELLE PARTITE DI TEST ---
    # Ogni iterazione crea una partita indipendente.
    for i in range(games):
        env = DamaEnv(opponent=opponent, seed=i)
        obs, info = env.reset(seed=i)
        done = False
        truncated = False
        # --- SOTTOSEZIONE: SIMULAZIONE DI UNA PARTITA ---
        # Il modello sceglie azioni finche' la partita termina o viene troncata.
        while not (done or truncated):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, truncated, info = env.step(int(action))
        # --- SOTTOSEZIONE: REGISTRAZIONE ESITO ---
        # Dopo la partita controlla chi ha vinto e aggiorna i contatori.
        winner = env.game.winner()
        if winner == WHITE:
            results["white_win"] += 1
        elif winner == BLACK:
            results["black_win"] += 1
        else:
            results["draw_or_truncated"] += 1
    return results


# === SEZIONE: ESECUZIONE SICURA DELLA VALUTAZIONE ===
# SCOPO: evitare errore se model non esiste perche' non hai ancora eseguito il training.
try:
    print(evaluate_model(model, games=10, opponent="heuristic"))
except NameError:
    print("Esegui prima la cella di training o carica un modello salvato.")


{'white_win': 0, 'black_win': 0, 'draw_or_truncated': 10}


In [ ]:
# === SEZIONE: CONVERSIONE INPUT TESTUALE IN MOSSA ===
# SCOPO: leggere una stringa tipo b6-a5 o b6xd4 e trasformarla in una Move legale.
def user_move_from_text(game: DamaGame, text: str, player: int) -> Move:
    # Esempi validi: b6-a5, b6xd4xf2. Conta start/end; il motore sceglie la sequenza legale corrispondente.
    # --- SOTTOSEZIONE: PULIZIA E DIVISIONE DEL TESTO ---
    # Trasforma sia "x" sia "-" nello stesso separatore, poi divide in caselle.
    tokens = text.replace('x', '-').replace(' ', '').split('-')
    if len(tokens) < 2:
        raise ValueError("Inserisci una mossa tipo b6-a5 oppure b6xd4.")
    start = parse_square(tokens[0])
    end = parse_square(tokens[-1])
    # --- SOTTOSEZIONE: RICERCA MOSSA TRA QUELLE LEGALI ---
    # Non basta che il testo sembri valido: la mossa deve esistere tra le mosse legali correnti.
    candidates = [m for m in game.legal_moves(player, allow_soffio=True) if m.start == start and m.end == end]
    if not candidates:
        legal = ', '.join(str(m) for m in game.legal_moves(player, allow_soffio=True))
        raise ValueError(f"Mossa non legale. Mosse disponibili: {legal}")
    # --- SOTTOSEZIONE: SCELTA DELLA SEQUENZA MIGLIORE ---
    # Se ci sono piu' mosse con stessa partenza/arrivo, prende quella con piu' catture.
    return max(candidates, key=lambda m: len(m.captures))


# === SEZIONE: PARTITA TESTUALE CONTRO LA MACCHINA ===
# SCOPO: permettere una partita scrivendo mosse da tastiera.
# Non e' grafica: stampa la scacchiera come testo.
def play_vs_master(master_depth: int = 6, human_player: int = WHITE):
    # --- SOTTOSEZIONE: SETUP PARTITA ---
    # Crea una nuova partita e decide quale colore ha la macchina.
    game = DamaGame(current_player=WHITE)
    machine = -human_player
    print(game.render_text())
    print("Tu giochi con", "BIANCO" if human_player == WHITE else "NERO")

    # --- SOTTOSEZIONE: CICLO DEI TURNI ---
    # Continua finche' non c'e' un vincitore e non si supera il limite massimo di mosse.
    while game.winner() is None and len(game.history) < MAX_GAME_STEPS:
        # PASSAGGIO: se e' il turno umano, chiede una mossa con input().
        if game.current_player == human_player:
            legal = game.legal_moves(human_player)
            print("Mosse legali:", ', '.join(str(m) for m in legal))
            text = input("La tua mossa: ")
            try:
                move = user_move_from_text(game, text, human_player)
                blown = game.apply_move(move, allow_soffio=True)
                if blown is not None:
                    print("Soffio:", square_name(*blown))
            except ValueError as exc:
                print(exc)
                continue
        # PASSAGGIO: se e' il turno della macchina, usa minimax_move.
        else:
            move = minimax_move(game, machine, depth=master_depth)
            if move is None:
                break
            print("Macchina:", move)
            game.apply_move(move)
        print(game.render_text())

    # --- SOTTOSEZIONE: RISULTATO FINALE ---
    # Dopo il ciclo stampa chi ha vinto.
    winner = game.winner()
    if winner == human_player:
        print("Hai vinto. Aumenta master_depth o migliora la valutazione euristica.")
    elif winner == machine:
        print("Ha vinto la macchina.")
    else:
        print("Partita terminata senza vincitore.")


# Avvia una partita interattiva quando vuoi.
play_vs_master(master_depth=6, human_player=WHITE)


   a b c d e f g h
8    b   b   b   b  8
7  b   b   b   b    7
6    b   b   b   b  6
5  .   .   .   .    5
4    .   .   .   .  4
3  w   w   w   w    3
2    w   w   w   w  2
1  w   w   w   w    1
   a b c d e f g h
Tu giochi con BIANCO
Mosse legali: a3-b4, c3-b4, c3-d4, e3-d4, e3-f4, g3-f4, g3-h4
Inserisci una mossa tipo b6-a5 oppure b6xd4.
Mosse legali: a3-b4, c3-b4, c3-d4, e3-d4, e3-f4, g3-f4, g3-h4
Inserisci una mossa tipo b6-a5 oppure b6xd4.
Mosse legali: a3-b4, c3-b4, c3-d4, e3-d4, e3-f4, g3-f4, g3-h4
Inserisci una mossa tipo b6-a5 oppure b6xd4.
Mosse legali: a3-b4, c3-b4, c3-d4, e3-d4, e3-f4, g3-f4, g3-h4


<!-- VISIONE_CAMERA_DAMA_V1 -->
## Visione con Telecamera

Questa sezione aggiunge una modalita' per giocare su una scacchiera fisica usando una webcam fissa dall'alto.

Flusso V1:
1. catturi un frame dalla webcam;
2. inserisci i 4 angoli della scacchiera;
3. il notebook corregge la prospettiva;
4. impara i colori dalla posizione iniziale;
5. legge la scacchiera fisica;
6. valida la mossa umana con `DamaGame`;
7. calcola la risposta della macchina con Master minimax o PPO;
8. ti dice quale pezzo muovere fisicamente per la macchina.

Questa versione e' semi-automatica: se la lettura e' ambigua, non forza lo stato interno ma chiede verifica/correzione.


In [19]:
# === SEZIONE: MODULO VISIONE CAMERA PER DAMA FISICA ===
# SCOPO: collegare una webcam fissa dall'alto al motore DamaGame.
# Questa cella non avvia automaticamente la camera: definisce funzioni, classi e UI.

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    import cv2
    CV2_AVAILABLE = True
except ImportError:
    cv2 = None
    CV2_AVAILABLE = False


# === SEZIONE: CONFIGURAZIONE VISIONE ===
# BOARD_WARP_SIZE e' la dimensione dell'immagine raddrizzata della scacchiera.
# 640 significa 80 pixel per casella, perche' 640 / 8 = 80.
BOARD_WARP_SIZE = 640
CELL_PIXELS = BOARD_WARP_SIZE // BOARD_SIZE


@dataclass
class CameraCalibration:
    """Salva l'omografia che trasforma il frame camera in una scacchiera vista dall'alto."""
    corners: np.ndarray
    matrix: np.ndarray
    board_size_px: int = BOARD_WARP_SIZE

    def warp(self, frame_bgr: np.ndarray) -> np.ndarray:
        if not CV2_AVAILABLE:
            raise RuntimeError("OpenCV non e' installato. Installa opencv-python.")
        return cv2.warpPerspective(frame_bgr, self.matrix, (self.board_size_px, self.board_size_px))


@dataclass
class PieceColorProfiles:
    """Colori medi imparati dalla posizione iniziale."""
    white_bgr: np.ndarray
    black_bgr: np.ndarray
    empty_bgr: np.ndarray
    max_distance: float = 72.0


@dataclass
class BoardReadResult:
    """Risultato di una lettura camera."""
    board: np.ndarray
    confidence: float
    ambiguous: list
    warped_bgr: np.ndarray
    metrics: dict


# === SEZIONE: ACQUISIZIONE FRAME ===
def capture_camera_frame(camera_index: int = 0, width: Optional[int] = None, height: Optional[int] = None) -> np.ndarray:
    """Cattura un singolo frame BGR dalla webcam."""
    if not CV2_AVAILABLE:
        raise RuntimeError("OpenCV non e' installato. Esegui: %pip install opencv-python")
    cap = cv2.VideoCapture(int(camera_index))
    if width:
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, int(width))
    if height:
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, int(height))
    ok, frame = cap.read()
    cap.release()
    if not ok or frame is None:
        raise RuntimeError(f"Impossibile leggere dalla camera index={camera_index}.")
    return frame


def show_bgr_image(frame_bgr: np.ndarray, title: str = "Frame", figsize=(7, 7), grid: bool = False) -> None:
    """Mostra un'immagine BGR dentro il notebook."""
    if not CV2_AVAILABLE:
        raise RuntimeError("OpenCV non e' installato.")
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(rgb)
    plt.title(title)
    plt.axis("off")
    if grid:
        step = frame_bgr.shape[0] / BOARD_SIZE
        for i in range(1, BOARD_SIZE):
            plt.axhline(i * step, color="yellow", linewidth=0.8)
            plt.axvline(i * step, color="yellow", linewidth=0.8)
    plt.show()


# === SEZIONE: CALIBRAZIONE PROSPETTICA ===
def parse_corner_text(text: str) -> np.ndarray:
    """Legge angoli scritti come: x1,y1; x2,y2; x3,y3; x4,y4."""
    pairs = []
    for chunk in text.replace("\n", ";").split(";"):
        chunk = chunk.strip()
        if not chunk:
            continue
        x_str, y_str = chunk.split(",")
        pairs.append((float(x_str.strip()), float(y_str.strip())))
    if len(pairs) != 4:
        raise ValueError("Servono esattamente 4 angoli nel formato x,y; x,y; x,y; x,y")
    return np.array(pairs, dtype=np.float32)


def order_corners(points: np.ndarray) -> np.ndarray:
    """Ordina i 4 punti in alto-sinistra, alto-destra, basso-destra, basso-sinistra."""
    pts = np.asarray(points, dtype=np.float32)
    sums = pts.sum(axis=1)
    diffs = np.diff(pts, axis=1).reshape(-1)
    ordered = np.zeros((4, 2), dtype=np.float32)
    ordered[0] = pts[np.argmin(sums)]
    ordered[2] = pts[np.argmax(sums)]
    ordered[1] = pts[np.argmin(diffs)]
    ordered[3] = pts[np.argmax(diffs)]
    return ordered


def build_calibration(corners: np.ndarray, board_size_px: int = BOARD_WARP_SIZE) -> CameraCalibration:
    """Costruisce la matrice prospettica dai 4 angoli della scacchiera."""
    if not CV2_AVAILABLE:
        raise RuntimeError("OpenCV non e' installato.")
    ordered = order_corners(corners)
    dst = np.array(
        [[0, 0], [board_size_px - 1, 0], [board_size_px - 1, board_size_px - 1], [0, board_size_px - 1]],
        dtype=np.float32,
    )
    matrix = cv2.getPerspectiveTransform(ordered, dst)
    return CameraCalibration(corners=ordered, matrix=matrix, board_size_px=board_size_px)


# === SEZIONE: ANALISI DELLE CASELLE ===
def cell_crop(warped_bgr: np.ndarray, row: int, col: int, margin_ratio: float = 0.23) -> np.ndarray:
    """Estrae la parte centrale di una casella, evitando bordi e linee della scacchiera."""
    cell = warped_bgr.shape[0] // BOARD_SIZE
    y0, y1 = row * cell, (row + 1) * cell
    x0, x1 = col * cell, (col + 1) * cell
    margin = int(cell * margin_ratio)
    return warped_bgr[y0 + margin:y1 - margin, x0 + margin:x1 - margin]


def mean_bgr(crop: np.ndarray) -> np.ndarray:
    """Calcola il colore medio BGR di un ritaglio."""
    return crop.reshape(-1, 3).mean(axis=0)


def learn_piece_profiles_from_initial(warped_bgr: np.ndarray) -> PieceColorProfiles:
    """Impara colori medi assumendo che la scacchiera fisica sia nella posizione iniziale."""
    white_samples = []
    black_samples = []
    empty_samples = []
    for row in range(BOARD_SIZE):
        for col in range(BOARD_SIZE):
            if not playable(row, col):
                continue
            sample = mean_bgr(cell_crop(warped_bgr, row, col))
            if row < 3:
                black_samples.append(sample)
            elif row > 4:
                white_samples.append(sample)
            else:
                empty_samples.append(sample)
    if not white_samples or not black_samples or not empty_samples:
        raise RuntimeError("Impossibile imparare profili colore dalla posizione iniziale.")
    return PieceColorProfiles(
        white_bgr=np.mean(white_samples, axis=0),
        black_bgr=np.mean(black_samples, axis=0),
        empty_bgr=np.mean(empty_samples, axis=0),
    )


def classify_cell(crop: np.ndarray, profiles: Optional[PieceColorProfiles]) -> Tuple[int, float, dict]:
    """Classifica una casella come vuota, bianca o nera."""
    color = mean_bgr(crop)
    gray_std = float(np.std(crop.mean(axis=2)))
    metrics = {"mean_bgr": color, "gray_std": gray_std}

    if profiles is not None:
        d_white = float(np.linalg.norm(color - profiles.white_bgr))
        d_black = float(np.linalg.norm(color - profiles.black_bgr))
        d_empty = float(np.linalg.norm(color - profiles.empty_bgr))
        distances = {WHITE_MAN: d_white, BLACK_MAN: d_black, EMPTY: d_empty}
        piece = min(distances, key=distances.get)
        ordered = sorted(distances.values())
        margin = ordered[1] - ordered[0] if len(ordered) > 1 else 0.0
        confidence = max(0.0, min(1.0, margin / profiles.max_distance))
        metrics.update({"d_white": d_white, "d_black": d_black, "d_empty": d_empty})
        return piece, confidence, metrics

    # Fallback senza profili: funziona solo con pezzi molto contrastati.
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV) if CV2_AVAILABLE else None
    value = float(hsv[:, :, 2].mean()) if hsv is not None else float(color.mean())
    if value > 175:
        return WHITE_MAN, 0.55, metrics
    if value < 75:
        return BLACK_MAN, 0.55, metrics
    return EMPTY, 0.35, metrics


def preserve_engine_kings(detected_board: np.ndarray, engine_board: np.ndarray) -> np.ndarray:
    """Mantiene le dame note al motore quando la camera vede solo il colore del pezzo."""
    out = detected_board.copy()
    for row in range(BOARD_SIZE):
        for col in range(BOARD_SIZE):
            if owner(int(out[row, col])) and owner(int(out[row, col])) == owner(int(engine_board[row, col])):
                if is_king(int(engine_board[row, col])):
                    out[row, col] = engine_board[row, col]
    return out


def read_physical_board(
    frame_bgr: np.ndarray,
    calibration: CameraCalibration,
    profiles: Optional[PieceColorProfiles] = None,
    engine_board: Optional[np.ndarray] = None,
    min_confidence: float = 0.16,
) -> BoardReadResult:
    """Legge la scacchiera fisica dal frame camera."""
    warped = calibration.warp(frame_bgr)
    board = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=np.int8)
    ambiguous = []
    metrics = {}
    confidences = []
    for row in range(BOARD_SIZE):
        for col in range(BOARD_SIZE):
            if not playable(row, col):
                continue
            piece, conf, cell_metrics = classify_cell(cell_crop(warped, row, col), profiles)
            board[row, col] = piece
            confidences.append(conf)
            metrics[(row, col)] = cell_metrics
            if conf < min_confidence:
                ambiguous.append((row, col, conf))
    if engine_board is not None:
        board = preserve_engine_kings(board, engine_board)
    confidence = float(np.mean(confidences)) if confidences else 0.0
    return BoardReadResult(board=board, confidence=confidence, ambiguous=ambiguous, warped_bgr=warped, metrics=metrics)


# === SEZIONE: CONFRONTO TRA CAMERA E MOTORE ===
def owner_board(board: np.ndarray) -> np.ndarray:
    """Riduce la board a soli proprietari: 1 bianco, -1 nero, 0 vuoto."""
    return np.vectorize(lambda x: owner(int(x)))(board).astype(np.int8)


def board_after_move_without_soffio(board: np.ndarray, move: Move) -> np.ndarray:
    """Simula solo spostamento/catture della mossa, senza applicare il soffio."""
    next_board = board.astype(np.int8).copy()
    sr, sc = move.start
    er, ec = move.end
    piece = int(next_board[sr, sc])
    next_board[sr, sc] = EMPTY
    for cr, cc in move.captures:
        next_board[cr, cc] = EMPTY
    next_board[er, ec] = promote(piece, er)
    return next_board


def infer_move_from_camera(game: DamaGame, observed_board: np.ndarray, player: int, allow_soffio: bool = True) -> Tuple[Optional[Move], list]:
    """Trova quale mossa legale produce la board osservata dalla camera."""
    observed_owners = owner_board(observed_board)
    matches = []
    for move in game.legal_moves(player, allow_soffio=allow_soffio):
        expected = board_after_move_without_soffio(game.board, move)
        mismatch = int(np.sum(owner_board(expected) != observed_owners))
        if mismatch == 0:
            matches.append((move, mismatch))
    if len(matches) == 1:
        return matches[0][0], matches
    return None, matches


def board_mismatches(expected_board: np.ndarray, observed_board: np.ndarray) -> List[str]:
    """Restituisce le caselle dove camera e motore non coincidono per proprietario."""
    expected = owner_board(expected_board)
    observed = owner_board(observed_board)
    out = []
    for row in range(BOARD_SIZE):
        for col in range(BOARD_SIZE):
            if playable(row, col) and expected[row, col] != observed[row, col]:
                out.append(square_name(row, col))
    return out


# === SEZIONE: MOTORE MACCHINA, MASTER O PPO ===
def load_ppo_model_if_available(path: str = "dama_ppo_model.zip") -> Optional[Any]:
    """Carica PPO solo se stable-baselines3 e il file modello sono disponibili."""
    model_path = Path(path)
    if not model_path.exists():
        return None
    try:
        from stable_baselines3 import PPO
        return PPO.load(str(model_path.with_suffix("")))
    except Exception as exc:
        print(f"PPO non caricato: {exc}")
        return None


def ppo_move_for_game(game: DamaGame, model: Any, player: int) -> Optional[Move]:
    """Usa PPO per scegliere una mossa. Se PPO non e' adatto, torna None."""
    if model is None:
        return None
    obs = game.board.copy()
    if player == BLACK:
        # Il PPO del notebook e' stato pensato per il bianco: invertiamo la board per riusarlo sul nero.
        obs = -np.flipud(game.board.copy())
    action, _ = model.predict(obs, deterministic=True)
    start, end = DamaEnv.decode_action(int(action))
    if player == BLACK:
        start = (BOARD_SIZE - 1 - start[0], start[1])
        end = (BOARD_SIZE - 1 - end[0], end[1])
    candidates = [m for m in game.legal_moves(player) if m.start == start and m.end == end]
    if not candidates:
        return None
    return max(candidates, key=lambda m: len(m.captures))


def select_machine_move(game: DamaGame, engine: str, master_depth: int, ppo_model: Optional[Any]) -> Optional[Move]:
    """Sceglie la mossa macchina con Master o PPO, con fallback sicuro su Master."""
    player = game.current_player
    if engine == "PPO":
        move = ppo_move_for_game(game, ppo_model, player)
        if move is not None:
            return move
    return minimax_move(game, player, depth=int(master_depth))


# === SEZIONE: SESSIONE INTERATTIVA CAMERA ===
class PhysicalDamaCameraSession:
    """Gestisce partita fisica, camera, calibrazione, lettura board e motore macchina."""

    def __init__(self, camera_index: int = 0):
        self.camera_index = camera_index
        self.game = DamaGame(current_player=WHITE)
        self.calibration: Optional[CameraCalibration] = None
        self.profiles: Optional[PieceColorProfiles] = None
        self.last_frame: Optional[np.ndarray] = None
        self.last_read: Optional[BoardReadResult] = None
        self.pending_machine_move: Optional[Move] = None
        self.ppo_model: Optional[Any] = None

        self.camera_input = widgets.IntText(value=camera_index, description="Camera")
        self.corners_input = widgets.Textarea(
            value="",
            description="Angoli",
            placeholder="x1,y1; x2,y2; x3,y3; x4,y4",
            layout=widgets.Layout(width="520px", height="70px"),
        )
        self.engine_dropdown = widgets.Dropdown(options=["Master", "PPO"], value="Master", description="Motore")
        self.depth_slider = widgets.IntSlider(value=5, min=2, max=8, step=1, description="Depth")
        self.status = widgets.HTML(value="<b>Pronto.</b> Cattura un frame e calibra i 4 angoli.")
        self.output = widgets.Output()

        self.capture_button = widgets.Button(description="Cattura frame", icon="camera")
        self.calibrate_button = widgets.Button(description="Calibra angoli", icon="crosshairs")
        self.learn_button = widgets.Button(description="Impara colori iniziali", icon="eyedropper")
        self.read_button = widgets.Button(description="Leggi scacchiera", icon="eye")
        self.human_button = widgets.Button(description="Valida mossa umana", icon="check", button_style="primary")
        self.verify_button = widgets.Button(description="Verifica mossa macchina", icon="search")
        self.reset_button = widgets.Button(description="Nuova partita", icon="refresh")
        self.ppo_button = widgets.Button(description="Carica PPO", icon="upload")

        self.capture_button.on_click(self._on_capture)
        self.calibrate_button.on_click(self._on_calibrate)
        self.learn_button.on_click(self._on_learn_profiles)
        self.read_button.on_click(self._on_read_board)
        self.human_button.on_click(self._on_human_move)
        self.verify_button.on_click(self._on_verify_machine)
        self.reset_button.on_click(self._on_reset)
        self.ppo_button.on_click(self._on_load_ppo)

    def show(self):
        controls_1 = widgets.HBox([self.camera_input, self.capture_button, self.calibrate_button, self.learn_button])
        controls_2 = widgets.HBox([self.engine_dropdown, self.depth_slider, self.ppo_button, self.reset_button])
        controls_3 = widgets.HBox([self.read_button, self.human_button, self.verify_button])
        display(widgets.VBox([self.status, self.corners_input, controls_1, controls_2, controls_3, self.output]))
        return self

    def _set_status(self, message: str):
        self.status.value = message

    def capture(self) -> np.ndarray:
        self.camera_index = int(self.camera_input.value)
        self.last_frame = capture_camera_frame(self.camera_index)
        return self.last_frame

    def read_board_from_camera(self) -> BoardReadResult:
        if self.calibration is None:
            raise RuntimeError("Prima calibra i 4 angoli della scacchiera.")
        frame = self.capture()
        self.last_read = read_physical_board(frame, self.calibration, self.profiles, self.game.board)
        return self.last_read

    def _on_capture(self, _button):
        with self.output:
            clear_output(wait=True)
            try:
                frame = self.capture()
                show_bgr_image(frame, "Frame camera: inserisci i 4 angoli nel box Angoli")
                self._set_status("Frame catturato. Inserisci i 4 angoli come x,y; x,y; x,y; x,y.")
            except Exception as exc:
                self._set_status(f"<b>Errore camera:</b> {exc}")

    def _on_calibrate(self, _button):
        with self.output:
            clear_output(wait=True)
            try:
                if self.last_frame is None:
                    self.capture()
                corners = parse_corner_text(self.corners_input.value)
                self.calibration = build_calibration(corners)
                warped = self.calibration.warp(self.last_frame)
                show_bgr_image(warped, "Scacchiera raddrizzata", grid=True)
                self._set_status("Calibrazione completata. Ora metti la posizione iniziale e premi 'Impara colori iniziali'.")
            except Exception as exc:
                self._set_status(f"<b>Errore calibrazione:</b> {exc}")

    def _on_learn_profiles(self, _button):
        with self.output:
            clear_output(wait=True)
            try:
                if self.calibration is None:
                    raise RuntimeError("Prima calibra la scacchiera.")
                frame = self.capture()
                warped = self.calibration.warp(frame)
                self.profiles = learn_piece_profiles_from_initial(warped)
                show_bgr_image(warped, "Profili colore imparati dalla posizione iniziale", grid=True)
                self._set_status("Profili colore salvati. Ora puoi muovere fisicamente e validare la mossa umana.")
            except Exception as exc:
                self._set_status(f"<b>Errore profili colore:</b> {exc}")

    def _on_read_board(self, _button):
        with self.output:
            clear_output(wait=True)
            try:
                result = self.read_board_from_camera()
                show_bgr_image(result.warped_bgr, f"Board letta - confidence {result.confidence:.2f}", grid=True)
                print(DamaGame(result.board).render_text())
                if result.ambiguous:
                    print("Caselle ambigue:", [(square_name(r, c), round(conf, 2)) for r, c, conf in result.ambiguous[:12]])
                self._set_status(f"Lettura completata. Confidence media: {result.confidence:.2f}")
            except Exception as exc:
                self._set_status(f"<b>Errore lettura:</b> {exc}")

    def _on_human_move(self, _button):
        with self.output:
            clear_output(wait=True)
            try:
                if self.game.current_player != WHITE:
                    raise RuntimeError("In questa V1 l'umano gioca il bianco. Verifica prima la mossa macchina se necessario.")
                result = self.read_board_from_camera()
                if result.ambiguous:
                    show_bgr_image(result.warped_bgr, "Lettura ambigua", grid=True)
                    self._set_status("Lettura ambigua: correggi luce/camera o ricalibra prima di aggiornare il motore.")
                    print("Caselle ambigue:", [(square_name(r, c), round(conf, 2)) for r, c, conf in result.ambiguous[:16]])
                    return
                move, matches = infer_move_from_camera(self.game, result.board, WHITE, allow_soffio=True)
                if move is None:
                    show_bgr_image(result.warped_bgr, "Mossa non riconosciuta", grid=True)
                    self._set_status("Mossa umana non riconosciuta o ambigua: il motore NON e' stato aggiornato.")
                    print("Match trovati:", [(str(m), diff) for m, diff in matches])
                    print("Stato camera:")
                    print(DamaGame(result.board).render_text())
                    return
                blown = self.game.apply_move(move, allow_soffio=True)
                print("Mossa umana valida:", move)
                if blown is not None:
                    print("Soffio: rimuovi fisicamente la pedina in", square_name(*blown))
                machine_move = select_machine_move(self.game, self.engine_dropdown.value, self.depth_slider.value, self.ppo_model)
                if machine_move is None:
                    self._set_status("Nessuna mossa macchina disponibile. Partita finita o stato anomalo.")
                    return
                self.game.apply_move(machine_move)
                self.pending_machine_move = machine_move
                print("Macchina:", machine_move)
                if machine_move.captures:
                    print("Rimuovi le pedine catturate:", ", ".join(square_name(*sq) for sq in machine_move.captures))
                self._set_status(f"Mossa umana {move}. Ora muovi fisicamente la macchina: <b>{machine_move}</b>, poi premi Verifica.")
            except Exception as exc:
                self._set_status(f"<b>Errore mossa umana:</b> {exc}")

    def _on_verify_machine(self, _button):
        with self.output:
            clear_output(wait=True)
            try:
                if self.pending_machine_move is None:
                    self._set_status("Non c'e' una mossa macchina in attesa di verifica.")
                    return
                result = self.read_board_from_camera()
                mismatches = board_mismatches(self.game.board, result.board)
                show_bgr_image(result.warped_bgr, "Verifica mossa macchina", grid=True)
                if mismatches:
                    self._set_status("La scacchiera fisica non coincide ancora con il motore.")
                    print("Caselle da controllare:", ", ".join(mismatches))
                    print("Mossa attesa macchina:", self.pending_machine_move)
                    return
                self.pending_machine_move = None
                self._set_status("Mossa macchina verificata. Tocca di nuovo all'umano.")
                print("Stato ufficiale:")
                print(self.game.render_text())
            except Exception as exc:
                self._set_status(f"<b>Errore verifica:</b> {exc}")

    def _on_reset(self, _button):
        self.game = DamaGame(current_player=WHITE)
        self.pending_machine_move = None
        self.last_read = None
        self._set_status("Nuova partita creata. Se la camera e i colori non sono cambiati, puoi ripartire dalla validazione mosse.")
        with self.output:
            clear_output(wait=True)
            print(self.game.render_text())

    def _on_load_ppo(self, _button):
        with self.output:
            clear_output(wait=True)
            self.ppo_model = load_ppo_model_if_available("dama_ppo_model.zip")
            if self.ppo_model is None:
                self._set_status("PPO non disponibile: usero' il fallback Master quando serve.")
            else:
                self._set_status("PPO caricato. Puoi selezionare Motore = PPO.")


def play_physical_camera_dama(camera_index: int = 0) -> PhysicalDamaCameraSession:
    """Avvia la UI per giocare con scacchiera fisica e webcam."""
    session = PhysicalDamaCameraSession(camera_index=camera_index)
    return session.show()


<!-- VISIONE_CAMERA_DAMA_V1 -->
### Avvio rapido della modalita' camera

Esegui la cella sotto dopo aver eseguito il motore `DamaGame` e la cella del modulo visione.

Passi consigliati:
1. premi **Cattura frame**;
2. scrivi i 4 angoli della scacchiera nel formato `x,y; x,y; x,y; x,y`;
3. premi **Calibra angoli**;
4. metti la scacchiera nella posizione iniziale e premi **Impara colori iniziali**;
5. muovi il bianco fisicamente e premi **Valida mossa umana**;
6. esegui fisicamente la mossa indicata dalla macchina e premi **Verifica mossa macchina**.


In [21]:
# === SEZIONE: AVVIO UI CAMERA ===
# Esegui questa cella per aprire i controlli della modalita' fisica con webcam.
# Se hai piu' webcam, cambia camera_index a 1, 2, ...
physical_session = play_physical_camera_dama(camera_index=0)


## Come rendere la modalita' Master piu' forte

1. Aumenta `master_depth` a 7 o 8 se il computer lo regge.
2. Allena il modello per molti piu' step, per esempio `200_000` o `1_000_000`.
3. Usa self-play: alterna avversari `random`, `heuristic` e `master`.
4. Migliora `positional_score` con regole strategiche: protezione dai bordi, mobilita', minacce di cattura, promozione imminente.
5. Per una vera dama competitiva, integra un motore alpha-beta piu' profondo con transposition table e ordinamento mosse avanzato.